In [15]:
!pip install "setuptools<71.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 931.1/931.1 kB 11.4 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0


In [19]:
import torch
from typing import Dict

from tdc.single_pred import Develop

import networkx as nx
import graphein.protein as gp
from graphein.ml.conversion import GraphFormatConvertor
from graphein.ml import InMemoryProteinGraphDataset, ProteinGraphDataset

In [3]:
data = Develop(name = 'SAbDab_Chen')
split = data.get_split()
split["train"].head()

Found local copy...
Loading...
Done!


,Antibody_ID,Antibody,Y
0,12e8,['EVQLQQSGAEVVRSGASVKLSCTASGFNIKDYYIHWVKQRPEKG...,0
1,15c8,['EVQLQQSGAELVKPGASVKLSCTASGFNIKDTYMHWVKQKPEQG...,0
2,1a0q,['EVQLQESDAELVKPGASVKISCKASGYTFTDHVIHWVKQKPEQG...,1
3,1a14,['QVQLQQSGAELVKPGASVRMSCKASGYTFTNYNMYWVKQSPGQG...,0
4,1a2y,['QVQLQESGPGLVAPSQSLSITCTVSGFSLTGYGVNWVRQPPGKG...,0


In [4]:
from graphein.protein.utils import get_obsolete_mapping

obs = get_obsolete_mapping()

train_obs = [t for t in split["train"]["Antibody_ID"] if t in obs.keys()]
valid_obs = [t for t in split["valid"]["Antibody_ID"] if t in obs.keys()]
test_obs = [t for t in split["test"]["Antibody_ID"] if t in obs.keys()]

print(train_obs)
print(valid_obs)
print(test_obs)

['1om3', '1zls', '1zlu', '1zlw', '2f5a', '3l5y', '3qot', '3rvv', '3rvw', '3rvx', '3wxw', '4nx3', '4pp2', '4x4y', '5e99', '5kmv', '5usi', '6erx']
[]
['3wxv', '1zlv', '1op3', '3qos']


In [5]:
# If you want, you can get the PDB IDs of the new structure that replaces the obsolete entry
print("Replacement PDBs: ", [obs[t] for t  in train_obs])

# However, in this instance we will simply remove the obsolete entries from the train and test sets.
split["train"] = split["train"].loc[~split["train"]["Antibody_ID"].isin(train_obs)]
split["test"] = split["test"].loc[~split["test"]["Antibody_ID"].isin(test_obs)]

Replacement PDBs:  ['6n32', '6msy', '6mub', '6mnf', '2pr4', '4ps4', '5i18', '5vpl', '5vpg', '5vph', '6ks1', '4web', '5vco', '6dn0', '5ihu', '5vzx', '6b9j', '6fxn']


In [6]:
# Convert labels to tensors
def get_label_map(split_name: str) -> Dict[str, torch.Tensor]:
    return dict(zip(split[split_name].Antibody_ID, split[split_name].Y.apply(torch.tensor)))

train_labels = get_label_map("train")
valid_labels = get_label_map("valid")
test_labels = get_label_map("test")

In [7]:
from functools import partial

config = gp.ProteinGraphConfig(
        node_metadata_functions=[gp.amino_acid_one_hot],
        edge_construction_functions=[partial(gp.add_distance_threshold, threshold=6, long_interaction_threshold=0)]
)


In [8]:
config.dict()

{'granularity': 'CA',
 'keep_hets': [],
 'insertions': True,
 'alt_locs': 'max_occupancy',
 'pdb_dir': None,
 'verbose': False,
 'exclude_waters': True,
 'deprotonate': False,
 'protein_df_processing_functions': None,
 'edge_construction_functions': [functools.partial(<function add_distance_threshold at 0x7f90678ee340>, threshold=6, long_interaction_threshold=0)],
 'node_metadata_functions': [<function graphein.protein.features.nodes.amino_acid.amino_acid_one_hot(n, d: Dict[str, Any], return_array: bool = True, allowable_set: Optional[List[str]] = None) -> Union[pandas.core.series.Series, numpy.ndarray]>],
 'edge_metadata_functions': None,
 'graph_metadata_functions': None,
 'get_contacts_config': None,
 'dssp_config': None}

In [9]:
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg", columns=["coords", "edge_index", "amino_acid_one_hot"])

In [21]:
graph = gp.construct_graph(pdb_code="1lds", config=config)

Output()

In [22]:
print(g.graph['config'])

granularity='CA' keep_hets=[] insertions=True alt_locs='max_occupancy' pdb_dir=None verbose=False exclude_waters=True deprotonate=False protein_df_processing_functions=None edge_construction_functions=[functools.partial(<function add_distance_threshold at 0x7f90678ee340>, threshold=6, long_interaction_threshold=0)] node_metadata_functions=[<function amino_acid_one_hot at 0x7f90676d4900>] edge_metadata_functions=None graph_metadata_functions=None get_contacts_config=None dssp_config=None


In [17]:
import inspect

inspect.getsource(InMemoryProteinGraphDataset)

'class InMemoryProteinGraphDataset(InMemoryDataset):\n    def __init__(\n        self,\n        root: str,\n        name: str,\n        paths: Optional[List[str]] = None,\n        pdb_codes: Optional[List[str]] = None,\n        uniprot_ids: Optional[List[str]] = None,\n        graph_label_map: Optional[Dict[str, torch.Tensor]] = None,\n        node_label_map: Optional[Dict[str, torch.Tensor]] = None,\n        chain_selection_map: Optional[Dict[str, List[str]]] = None,\n        graphein_config: ProteinGraphConfig = ProteinGraphConfig(),\n        graph_format_convertor: GraphFormatConvertor = GraphFormatConvertor(\n            src_format="nx", dst_format="pyg"\n        ),\n        graph_transformation_funcs: Optional[List[Callable]] = None,\n        transform: Optional[Callable] = None,\n        pdb_transform: Optional[List[Callable]] = None,\n        pre_transform: Optional[Callable] = None,\n        pre_filter: Optional[Callable] = None,\n        num_cores: int = 16,\n        af_versio

In [16]:
len(split["test"]["Antibody_ID"])

478

In [23]:
def fit(g: nx.Graph) -> nx.Graph:
    g_config = g.graph['config']
    g_pdb_code = g.graph['pdb_code']

    g.graph.clear()
    g.graph['config'] = g_config
    g.graph['pdb_code'] = g_pdb_code

    return g

In [26]:
graph = fit(graph)
graph

In [ ]:
train_ds = InMemoryProteinGraphDataset(
    root="./data/",
    name="train",
    pdb_codes=split["train"]["Antibody_ID"],
    graph_label_map=train_labels,
    graphein_config=config,
    graph_format_convertor=convertor
    graph_transformation_funcs=[fit]
    )

valid_ds = InMemoryProteinGraphDataset(
    root="./data/",
    name="valid",
    pdb_codes=split["valid"]["Antibody_ID"],
    graph_label_map=valid_labels,
    graphein_config=config,
    graph_format_convertor=convertor
    graph_transformation_funcs=[]
    )

test_ds = InMemoryProteinGraphDataset(
    root="./data/",
    name="test",
    pdb_codes=split["test"]["Antibody_ID"],
    graph_label_map=test_labels,
    graphein_config=config,
    graph_format_convertor=convertor
    graph_transformation_funcs=[]
    )

Constructing Graphs...



Processing...


  0%|          | 0/1668 [00:00<?, ?it/s]